# 03 — MediaPipe Holistic Landmark Extraction

**Goal.** Convert every video into a per-frame landmark array, the single raw input for both representation branches.

**Approach**
- MediaPipe Holistic **0.10.14**: `static_image_mode=False`, `model_complexity=1`, smoothing on, `refine_face_landmarks=False`, no segmentation, detection/tracking confidence 0.5. A fresh Holistic object per video; frames converted BGR → RGB.
- Each clip is saved as `float32 (T, 1629)`: face 468 `[0:1404]`, pose 33 `[1404:1503]`, left hand 21 `[1503:1566]`, right hand 21 `[1566:1629]`, each landmark with (x, y, z).
- Undetected modality blocks and non-finite values are written as 0. Arrays are shape- and finiteness-checked, then written atomically.
- A smoke test runs before the full job. The run is **resumable**: finished arrays are validated and skipped, so a time-limited session can be continued.

**Result.** 6,000 arrays, 806,159 decoded frames, 0 failures. Detection rates: pose 1.00, face 0.995, left hand 0.783, right hand 0.876. Hands are the weak point, which is why masks are kept downstream.

**Outputs.** `holistic_landmarks_1629/<shard>/<word>/<video>.npy` (folder = class label), plus `holistic_metadata_*.csv` (per-video FPS and frame count, used again in 09), `holistic_failures_*.csv` and `holistic_manifest_*.json`.

**Limitation.** The final run reused 5,306 arrays from an earlier run. The resume check validates shape and dtype but does not record the extractor version or config of those arrays, so uniform provenance is intended but not proven.

In [ ]:
# ============================================================
# CELL 1 - Install MediaPipe, then repair a version clash
# ============================================================
# MediaPipe needs an older "protobuf" library than Kaggle ships with.
# Installing it downgrades protobuf, which breaks Kaggle's TensorFlow.
# MediaPipe then refuses to load because it peeks at TensorFlow on the
# way in. We don't use TensorFlow here, so we tell MediaPipe to shrug
# that off. The red wall of pip warnings this prints is expected.

import importlib.metadata
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_MEDIAPIPE_VERSION = "0.10.14"

try:
    installed = importlib.metadata.version("mediapipe")
except importlib.metadata.PackageNotFoundError:
    installed = None

if installed != REQUIRED_MEDIAPIPE_VERSION:
    print(f"Installing mediapipe=={REQUIRED_MEDIAPIPE_VERSION} (current: {installed})")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        f"mediapipe=={REQUIRED_MEDIAPIPE_VERSION}",
    ])
else:
    print(f"MediaPipe {installed} is already installed.")

spec = importlib.util.find_spec("mediapipe")
if spec is None or not spec.origin:
    raise RuntimeError("MediaPipe is not importable. Is Internet switched On?")

target = Path(spec.origin).parent / "tasks/python/core/optional_dependencies.py"
text = target.read_text(encoding="utf-8")

if "except ModuleNotFoundError:" in text:
    target.write_text(
        text.replace("except ModuleNotFoundError:", "except Exception:"),
        encoding="utf-8",
    )
    print(f"Patched: {target}")
else:
    print(f"No patch needed: {target}")

from mediapipe.python.solutions import holistic as _check

print("MediaPipe Holistic import: OK")
del _check

In [ ]:
# ============================================================
# CELL 2 - Settings
# ============================================================
# You do not need to change anything here for a normal run.
import json
from pathlib import Path

CONFIG = {
    "video_root": "",                  # blank = find the videos automatically
    "input_root": "/kaggle/input",
    "output_base": "/kaggle/working/holistic_landmarks_1629",

    # All 6,000 videos in one run.
    "shard_count": 1,
    "shard_index": 0,

    # 3 was measured fastest on Kaggle's CPU. More workers did not help,
    # because MediaPipe already uses several cores by itself.
    "max_workers": 3,
    "auto_worker_cap": 4,
    "benchmark_videos": 12,

    # Kaggle kills the session dead at 12 hours. We run right up to the wall:
    # stop starting new videos at 11 h 26 m, let the last few finish, write
    # the summary, and leave ~30 min so Kaggle can actually save the output.
    # Being killed mid-save is how you lose the whole night's work.
    "max_run_hours": 11.6,
    "finalization_reserve_minutes": 10,

    # How pessimistic the time estimate is. 1.15 = assume 15% slower than
    # the benchmark measured.
    "runtime_safety_factor": 1.15,
    "output_size_safety_factor": 1.15,

    # False = print a verdict and run anyway, rather than refusing to start.
    # Whatever finishes is saved, and you can resume to do the rest.
    "abort_if_projected_too_long": False,

    "checkpoint_every": 25,
    "overwrite_existing": False,

    # Resume: attach the previous version's Output as an input and rerun.
    # Finished files are copied back in and skipped automatically.
    "auto_resume": True,
    "resume_source": "",

    "model_complexity": 1,
    "min_detection_confidence": 0.50,
    "min_tracking_confidence": 0.50,
}

CONFIG_PATH = Path("/kaggle/working/holistic_config.json")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print(json.dumps(CONFIG, indent=2))
print(f"\nSaved to: {CONFIG_PATH}")
print("\n>>> SINGLE RUN: all videos, one output folder <<<")

In [ ]:
# ============================================================
# CELL 3 - Write the extraction program to a file
# ============================================================
# The real work lives in a standalone .py file rather than in the
# notebook, because running many processes in parallel is only safe
# that way.
from pathlib import Path

RUNNER_PATH = Path("/kaggle/working/run_holistic_extraction.py")
RUNNER_CODE = '#!/usr/bin/env python3\n"""Kaggle-safe MediaPipe Holistic landmark extraction.\n\nKey properties:\n- Detects the real SignBD RGB video root automatically.\n- Uses a fresh MediaPipe Holistic graph for every video.\n- Benchmarks 1..N workers and selects the fastest measured worker count.\n- Uses a standalone script and spawn-based workers for notebook safety.\n- Writes each .npy atomically.\n- Supports deterministic sharding and cross-commit resume.\n- Stops early enough to let Kaggle save a successful partial output.\n"""\n\n# Limit common native numerical thread pools before importing heavy packages.\nimport os\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\nos.environ.setdefault("NUMEXPR_NUM_THREADS", "1")\nos.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nos.environ.setdefault("GLOG_minloglevel", "2")\n\nimport argparse\nimport concurrent.futures as cf\nimport gc\nimport inspect\nimport json\nimport math\nimport multiprocessing as multiprocessing\nimport shutil\nimport sys\nimport time\nimport traceback\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport functools as _functools\nfrom tqdm.auto import tqdm as _tqdm\n\n# Committed runs write to a log file, not a terminal. Refresh sparingly.\ntqdm = _functools.partial(_tqdm, mininterval=30.0, file=sys.stdout)\n\nVIDEO_EXTENSIONS = {\n    ".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpeg", ".mpg", ".m4v"\n}\n\nFACE_LANDMARKS = 468\nPOSE_LANDMARKS = 33\nLEFT_HAND_LANDMARKS = 21\nRIGHT_HAND_LANDMARKS = 21\nCOORDINATES_PER_LANDMARK = 3\nTOTAL_LANDMARKS = (\n    FACE_LANDMARKS + POSE_LANDMARKS + LEFT_HAND_LANDMARKS + RIGHT_HAND_LANDMARKS\n)\nFEATURES_PER_FRAME = TOTAL_LANDMARKS * COORDINATES_PER_LANDMARK\n\nFACE_SLICE = slice(0, FACE_LANDMARKS * 3)\nPOSE_SLICE = slice(FACE_SLICE.stop, FACE_SLICE.stop + POSE_LANDMARKS * 3)\nLEFT_HAND_SLICE = slice(POSE_SLICE.stop, POSE_SLICE.stop + LEFT_HAND_LANDMARKS * 3)\nRIGHT_HAND_SLICE = slice(\n    LEFT_HAND_SLICE.stop,\n    LEFT_HAND_SLICE.stop + RIGHT_HAND_LANDMARKS * 3,\n)\n\nassert TOTAL_LANDMARKS == 543\nassert FEATURES_PER_FRAME == 1629\nassert RIGHT_HAND_SLICE.stop == FEATURES_PER_FRAME\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef atomic_write_json(path: Path, payload: Dict[str, Any]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(f"{path.name}.partial.{os.getpid()}")\n    try:\n        with temporary.open("w", encoding="utf-8") as handle:\n            json.dump(payload, handle, indent=2, default=str)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(f"{path.name}.partial.{os.getpid()}")\n    try:\n        frame.to_csv(temporary, index=False)\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef atomic_save_npy(path: Path, array: np.ndarray) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(f"{path.name}.partial.{os.getpid()}")\n    try:\n        with temporary.open("wb") as handle:\n            np.save(handle, array, allow_pickle=False)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef discover_video_root(config: Dict[str, Any]) -> Path:\n    explicit = str(config.get("video_root", "")).strip()\n    if explicit:\n        path = Path(explicit)\n        if not path.is_dir():\n            raise FileNotFoundError(f"Configured video_root does not exist: {path}")\n        return path\n\n    # This is the exact structure visible in the user\'s Kaggle sidebar.\n    preferred = Path(\n        "/kaggle/input/signbd-dataset/SignBD-Word/SignBD-Word_RGB"\n    )\n    if preferred.is_dir():\n        return preferred\n\n    input_root = Path(str(config.get("input_root", "/kaggle/input")))\n    candidates = sorted(\n        {\n            path.resolve()\n            for path in input_root.rglob("SignBD-Word_RGB")\n            if path.is_dir()\n        },\n        key=lambda path: (len(path.parts), str(path)),\n    )\n\n    if not candidates:\n        raise FileNotFoundError(\n            "Could not find a directory named \'SignBD-Word_RGB\' under "\n            f"{input_root}. Attach the SignBD_Dataset input first."\n        )\n\n    if len(candidates) > 1:\n        print("Multiple SignBD-Word_RGB directories were found:")\n        for candidate in candidates:\n            print(f"  {candidate}")\n        print(f"Using the shortest candidate: {candidates[0]}")\n\n    return candidates[0]\n\n\ndef discover_videos(video_root: Path) -> List[Path]:\n    videos = sorted(\n        path\n        for path in video_root.rglob("*")\n        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS\n    )\n    if not videos:\n        raise RuntimeError(f"No supported videos were found under {video_root}")\n    return videos\n\n\ndef class_from_relative_path(relative_path: Path) -> str:\n    """The word/class is the folder DIRECTLY containing the video file.\n\n    The real layout is DATASET/<word>/<clip>.mp4, so taking the first\n    directory returned "DATASET" for all 6,000 videos - one class instead\n    of 200. The immediate parent is correct regardless of how many wrapper\n    folders sit above it.\n    """\n    return relative_path.parent.name or "unknown"\n\n\ndef valid_existing_sequence(path: Path) -> Optional[Dict[str, Any]]:\n    if not path.is_file():\n        return None\n    try:\n        sequence = np.load(path, mmap_mode="r", allow_pickle=False)\n        if (\n            sequence.ndim == 2\n            and sequence.shape[0] > 0\n            and sequence.shape[1] == FEATURES_PER_FRAME\n            and np.issubdtype(sequence.dtype, np.floating)\n        ):\n            result = {\n                "frames": int(sequence.shape[0]),\n                "features": int(sequence.shape[1]),\n                "dtype": str(sequence.dtype),\n            }\n            del sequence\n            return result\n    except Exception:\n        return None\n    return None\n\n\ndef landmark_list_to_xyz(\n    landmark_list: Any,\n    expected_landmarks: int,\n) -> np.ndarray:\n    expected_features = expected_landmarks * COORDINATES_PER_LANDMARK\n    if landmark_list is None:\n        return np.zeros(expected_features, dtype=np.float32)\n\n    xyz = np.asarray(\n        [(point.x, point.y, point.z) for point in landmark_list.landmark],\n        dtype=np.float32,\n    )\n    expected_shape = (expected_landmarks, COORDINATES_PER_LANDMARK)\n    if xyz.shape != expected_shape:\n        raise ValueError(\n            f"Unexpected landmark shape {xyz.shape}; expected {expected_shape}."\n        )\n    return np.nan_to_num(xyz, nan=0.0, posinf=0.0, neginf=0.0).reshape(-1)\n\n\ndef pose_visibility_mean(pose_landmarks: Any) -> float:\n    if pose_landmarks is None:\n        return float("nan")\n    values = np.asarray(\n        [point.visibility for point in pose_landmarks.landmark],\n        dtype=np.float32,\n    )\n    if values.shape != (POSE_LANDMARKS,):\n        return float("nan")\n    return float(np.nanmean(values))\n\n\ndef _process_video_once(task: Dict[str, Any]) -> Dict[str, Any]:\n    """Process one whole video with a brand-new Holistic graph."""\n    started = time.monotonic()\n    video_path = Path(task["video_path"])\n    output_path_text = str(task.get("output_path", ""))\n    output_path = Path(output_path_text) if output_path_text else None\n    save_output = bool(task.get("save_output", True))\n    detector_config = dict(task["detector_config"])\n\n    try:\n        # Imports happen inside the spawned worker. No detector exists in the\n        # parent process, and every video receives a fresh graph below.\n        from mediapipe.python.solutions import holistic as mp_holistic\n\n        cv2.setNumThreads(1)\n        capture = cv2.VideoCapture(str(video_path))\n        if hasattr(cv2, "CAP_PROP_OPEN_TIMEOUT_MSEC"):\n            capture.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, 15_000)\n        if hasattr(cv2, "CAP_PROP_READ_TIMEOUT_MSEC"):\n            capture.set(cv2.CAP_PROP_READ_TIMEOUT_MSEC, 15_000)\n\n        if not capture.isOpened():\n            raise RuntimeError(f"OpenCV could not open video: {video_path}")\n\n        declared_frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n        fps = float(capture.get(cv2.CAP_PROP_FPS))\n        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n\n        sequence: List[np.ndarray] = []\n        modality_counts = {\n            "face": 0,\n            "pose": 0,\n            "left_hand": 0,\n            "right_hand": 0,\n        }\n        visibility_values: List[float] = []\n\n        detector = mp_holistic.Holistic(\n            static_image_mode=False,\n            model_complexity=int(detector_config["model_complexity"]),\n            smooth_landmarks=True,\n            enable_segmentation=False,\n            smooth_segmentation=False,\n            refine_face_landmarks=False,\n            min_detection_confidence=float(\n                detector_config["min_detection_confidence"]\n            ),\n            min_tracking_confidence=float(\n                detector_config["min_tracking_confidence"]\n            ),\n        )\n\n        try:\n            while True:\n                success, frame = capture.read()\n                if not success:\n                    break\n\n                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n                rgb.flags.writeable = False\n                results = detector.process(rgb)\n\n                face = landmark_list_to_xyz(\n                    results.face_landmarks,\n                    FACE_LANDMARKS,\n                )\n                pose = landmark_list_to_xyz(\n                    results.pose_landmarks,\n                    POSE_LANDMARKS,\n                )\n                left_hand = landmark_list_to_xyz(\n                    results.left_hand_landmarks,\n                    LEFT_HAND_LANDMARKS,\n                )\n                right_hand = landmark_list_to_xyz(\n                    results.right_hand_landmarks,\n                    RIGHT_HAND_LANDMARKS,\n                )\n\n                features = np.concatenate(\n                    [face, pose, left_hand, right_hand]\n                ).astype(np.float32, copy=False)\n\n                if features.shape != (FEATURES_PER_FRAME,):\n                    raise ValueError(\n                        f"Frame feature shape was {features.shape}; "\n                        f"expected ({FEATURES_PER_FRAME},)."\n                    )\n\n                sequence.append(features)\n                modality_counts["face"] += int(results.face_landmarks is not None)\n                modality_counts["pose"] += int(results.pose_landmarks is not None)\n                modality_counts["left_hand"] += int(\n                    results.left_hand_landmarks is not None\n                )\n                modality_counts["right_hand"] += int(\n                    results.right_hand_landmarks is not None\n                )\n\n                visibility = pose_visibility_mean(results.pose_landmarks)\n                if np.isfinite(visibility):\n                    visibility_values.append(visibility)\n        finally:\n            capture.release()\n            detector.close()\n\n        if not sequence:\n            raise RuntimeError(f"No readable frames were decoded from {video_path}")\n\n        sequence_array = np.stack(sequence).astype(np.float32, copy=False)\n        if sequence_array.ndim != 2 or sequence_array.shape[1] != FEATURES_PER_FRAME:\n            raise ValueError(f"Unexpected sequence shape: {sequence_array.shape}")\n        if not np.isfinite(sequence_array).all():\n            raise ValueError("Sequence contains non-finite values after sanitisation.")\n\n        if save_output:\n            if output_path is None:\n                raise ValueError("save_output=True requires output_path")\n            atomic_save_npy(output_path, sequence_array)\n\n        frames = int(sequence_array.shape[0])\n        result = {\n            "ok": True,\n            "video_path": str(video_path),\n            "relative_video_path": task["relative_video_path"],\n            "class": task["class"],\n            "landmark_file": str(output_path) if output_path else "",\n            "landmark_file_relative": task.get("landmark_file_relative", ""),\n            "frames": frames,\n            "features": FEATURES_PER_FRAME,\n            "dtype": str(sequence_array.dtype),\n            "declared_frame_count": declared_frame_count,\n            "fps": fps,\n            "width": width,\n            "height": height,\n            "face_detection_rate": modality_counts["face"] / frames,\n            "pose_detection_rate": modality_counts["pose"] / frames,\n            "left_hand_detection_rate": modality_counts["left_hand"] / frames,\n            "right_hand_detection_rate": modality_counts["right_hand"] / frames,\n            "mean_pose_visibility": (\n                float(np.mean(visibility_values))\n                if visibility_values\n                else float("nan")\n            ),\n            "output_bytes": int(sequence_array.nbytes),\n            "elapsed_seconds": time.monotonic() - started,\n            "status": "processed" if save_output else "benchmark",\n        }\n        del sequence_array\n        gc.collect()\n        return result\n\n    except Exception as error:\n        return {\n            "ok": False,\n            "video_path": str(video_path),\n            "relative_video_path": task.get("relative_video_path", ""),\n            "class": task.get("class", ""),\n            "error_type": type(error).__name__,\n            "error_message": str(error),\n            "traceback": traceback.format_exc(limit=8),\n            "elapsed_seconds": time.monotonic() - started,\n            "status": "failed",\n        }\n\n\ndef worker_warmup() -> None:\n    """Absorb the first-use anomaly in a freshly spawned worker.\n\n    Benchmarking showed exactly one failed video per worker process -\n    each worker failed whichever task it happened to receive first.\n    Building and running a throwaway graph at process start means a real\n    video is never the first thing a worker touches. Failures here are\n    deliberately swallowed: a raising initializer would kill the pool.\n    """\n    try:\n        from mediapipe.python.solutions import holistic as mp_holistic\n\n        cv2.setNumThreads(1)\n        detector = mp_holistic.Holistic(\n            static_image_mode=False,\n            model_complexity=1,\n            smooth_landmarks=True,\n            enable_segmentation=False,\n            smooth_segmentation=False,\n            refine_face_landmarks=False,\n            min_detection_confidence=0.5,\n            min_tracking_confidence=0.5,\n        )\n        detector.process(np.zeros((256, 256, 3), dtype=np.uint8))\n        detector.close()\n    except Exception:\n        pass\n\n\ndef process_video_task(task: Dict[str, Any]) -> Dict[str, Any]:\n    """Run one video, retrying once before recording a failure.\n\n    A transient first-use failure inside a worker should cost one retry,\n    not one permanently missing video in the dataset.\n    """\n    result = _process_video_once(task)\n    if result.get("ok"):\n        return result\n\n    first_error = (\n        f"{result.get(\'error_type\')}: {result.get(\'error_message\')}"\n    )\n    time.sleep(0.5)\n    retry = _process_video_once(task)\n\n    if retry.get("ok"):\n        retry["recovered_after_error"] = first_error\n        return retry\n\n    retry["first_attempt_error"] = first_error\n    return retry\n\n\ndef executor_kwargs(worker_count: int) -> Dict[str, Any]:\n    kwargs: Dict[str, Any] = {\n        "max_workers": worker_count,\n        "mp_context": multiprocessing.get_context("spawn"),\n        "initializer": worker_warmup,\n    }\n    if "max_tasks_per_child" in inspect.signature(cf.ProcessPoolExecutor).parameters:\n        kwargs["max_tasks_per_child"] = 50\n    return kwargs\n\n\ndef make_task(\n    video_path: Path,\n    video_root: Path,\n    shard_output_root: Path,\n    detector_config: Dict[str, Any],\n    save_output: bool,\n) -> Dict[str, Any]:\n    relative = video_path.relative_to(video_root)\n    output_path = (shard_output_root / relative).with_suffix(".npy")\n    return {\n        "video_path": str(video_path),\n        "relative_video_path": relative.as_posix(),\n        "class": class_from_relative_path(relative),\n        "output_path": str(output_path) if save_output else "",\n        "landmark_file_relative": (\n            output_path.relative_to(Path("/kaggle/working")).as_posix()\n            if save_output\n            else ""\n        ),\n        "save_output": save_output,\n        "detector_config": detector_config,\n    }\n\n\ndef evenly_spaced_items(items: Sequence[Any], count: int) -> List[Any]:\n    if not items or count <= 0:\n        return []\n    count = min(count, len(items))\n    indexes = np.unique(\n        np.linspace(0, len(items) - 1, num=count, dtype=int)\n    )\n    return [items[int(index)] for index in indexes]\n\n\ndef benchmark_workers(\n    sample_tasks: List[Dict[str, Any]],\n    worker_candidates: List[int],\n) -> Tuple[int, List[Dict[str, Any]], Dict[str, Any]]:\n    reports: List[Dict[str, Any]] = []\n    best_report: Optional[Dict[str, Any]] = None\n\n    for workers in worker_candidates:\n        print(f"\\nBenchmarking with {workers} worker(s)...")\n        started = time.monotonic()\n        results: List[Dict[str, Any]] = []\n\n        try:\n            with cf.ProcessPoolExecutor(**executor_kwargs(workers)) as executor:\n                futures = [executor.submit(process_video_task, task) for task in sample_tasks]\n                for future in tqdm(\n                    cf.as_completed(futures),\n                    total=len(futures),\n                    desc=f"Benchmark {workers} worker(s)",\n                    unit="video",\n                ):\n                    results.append(future.result())\n        except Exception as error:\n            report = {\n                "workers": workers,\n                "ok": False,\n                "error": f"{type(error).__name__}: {error}",\n            }\n            reports.append(report)\n            print(report["error"])\n            continue\n\n        elapsed = time.monotonic() - started\n        successes = [result for result in results if result.get("ok")]\n        failures = [result for result in results if not result.get("ok")]\n\n        if not successes:\n            report = {\n                "workers": workers,\n                "ok": False,\n                "elapsed_seconds": elapsed,\n                "successful_videos": 0,\n                "failed_videos": len(failures),\n                "error": "No benchmark video completed successfully.",\n            }\n            reports.append(report)\n            continue\n\n        if failures:\n            print(f"  {len(failures)} benchmark video(s) failed:")\n            for failure in failures[:3]:\n                print(\n                    f"    {Path(failure[\'video_path\']).name}: "\n                    f"{failure.get(\'error_type\')}: "\n                    f"{failure.get(\'error_message\')}"\n                )\n\n        total_frames = sum(int(result["frames"]) for result in successes)\n        total_output_bytes = sum(int(result["output_bytes"]) for result in successes)\n        throughput = len(successes) / elapsed\n        report = {\n            "workers": workers,\n            "ok": True,\n            "elapsed_seconds": elapsed,\n            "successful_videos": len(successes),\n            "failed_videos": len(failures),\n            "videos_per_second": throughput,\n            "seconds_per_video_effective": elapsed / len(successes),\n            "frames_per_second_effective": total_frames / elapsed,\n            "mean_frames_per_video": total_frames / len(successes),\n            "mean_output_bytes_per_video": total_output_bytes / len(successes),\n        }\n        reports.append(report)\n        print(json.dumps(report, indent=2))\n\n        if best_report is None or report["videos_per_second"] > best_report["videos_per_second"]:\n            best_report = report\n\n    if best_report is None:\n        raise RuntimeError(\n            "Every worker-count benchmark failed. Review the benchmark errors above."\n        )\n\n    return int(best_report["workers"]), reports, best_report\n\n\ndef locate_resume_source(\n    config: Dict[str, Any],\n    shard_tag: str,\n    current_shard_root: Path,\n) -> Optional[Path]:\n    explicit = str(config.get("resume_source", "")).strip()\n    if explicit:\n        candidate = Path(explicit)\n        if not candidate.is_dir():\n            raise FileNotFoundError(f"resume_source does not exist: {candidate}")\n        return candidate\n\n    if not bool(config.get("auto_resume", True)):\n        return None\n\n    input_root = Path(str(config.get("input_root", "/kaggle/input")))\n    candidates = sorted(\n        {\n            path.resolve()\n            for path in input_root.rglob(shard_tag)\n            if path.is_dir()\n            and path.parent.name == "holistic_landmarks_1629"\n            and path.resolve() != current_shard_root.resolve()\n        },\n        key=str,\n    )\n\n    if not candidates:\n        return None\n    if len(candidates) > 1:\n        formatted = "\\n".join(f"  {path}" for path in candidates)\n        raise RuntimeError(\n            "Multiple prior shard outputs were found. Set resume_source explicitly:\\n"\n            f"{formatted}"\n        )\n    return candidates[0]\n\n\ndef copy_resume_output(source: Path, destination: Path) -> None:\n    print(f"Resuming from previous Kaggle output: {source}")\n    print(f"Copying prior completed arrays to: {destination}")\n    destination.mkdir(parents=True, exist_ok=True)\n    shutil.copytree(source, destination, dirs_exist_ok=True)\n\n\ndef checkpoint(\n    metadata_rows: List[Dict[str, Any]],\n    failure_rows: List[Dict[str, Any]],\n    metadata_path: Path,\n    failure_path: Path,\n    state_path: Path,\n    state: Dict[str, Any],\n) -> None:\n    metadata_frame = pd.DataFrame(metadata_rows)\n    failure_frame = pd.DataFrame(failure_rows)\n    atomic_write_csv(metadata_path, metadata_frame)\n    atomic_write_csv(failure_path, failure_frame)\n    atomic_write_json(state_path, state)\n\n\ndef run_extraction(\n    tasks: List[Dict[str, Any]],\n    workers: int,\n    stop_submitting_at: float,\n    checkpoint_every: int,\n    metadata_rows: List[Dict[str, Any]],\n    failure_rows: List[Dict[str, Any]],\n    metadata_path: Path,\n    failure_path: Path,\n    state_path: Path,\n    base_state: Dict[str, Any],\n) -> Tuple[int, bool]:\n    if not tasks:\n        return 0, False\n\n    submitted = 0\n    completed = 0\n    stopped_for_deadline = False\n    task_iterator = iter(tasks)\n    pending: Dict[cf.Future, Dict[str, Any]] = {}\n\n    def submit_one(executor: cf.ProcessPoolExecutor) -> bool:\n        nonlocal submitted\n        if time.monotonic() >= stop_submitting_at:\n            return False\n        try:\n            task = next(task_iterator)\n        except StopIteration:\n            return False\n        try:\n            pending[executor.submit(process_video_task, task)] = task\n        except Exception as error:\n            print(\n                f"Executor stopped accepting work "\n                f"({type(error).__name__}: {error}). "\n                "Finishing in-flight videos and checkpointing.",\n                flush=True,\n            )\n            return False\n        submitted += 1\n        return True\n\n    with cf.ProcessPoolExecutor(**executor_kwargs(workers)) as executor:\n        for _ in range(min(workers, len(tasks))):\n            if not submit_one(executor):\n                break\n\n        with tqdm(total=len(tasks), desc="Extracting landmarks", unit="video") as progress:\n            while pending:\n                done, _ = cf.wait(\n                    list(pending),\n                    timeout=30,\n                    return_when=cf.FIRST_COMPLETED,\n                )\n\n                if not done:\n                    if time.monotonic() >= stop_submitting_at:\n                        stopped_for_deadline = True\n                    continue\n\n                for future in done:\n                    task = pending.pop(future)\n                    try:\n                        result = future.result()\n                    except Exception as error:\n                        result = {\n                            "ok": False,\n                            "video_path": task["video_path"],\n                            "relative_video_path": task["relative_video_path"],\n                            "class": task["class"],\n                            "error_type": type(error).__name__,\n                            "error_message": str(error),\n                            "traceback": traceback.format_exc(limit=8),\n                            "status": "future_failed",\n                        }\n\n                    if result.get("ok"):\n                        result.pop("ok", None)\n                        metadata_rows.append(result)\n                    else:\n                        result.pop("ok", None)\n                        failure_rows.append(result)\n\n                    completed += 1\n                    progress.update(1)\n\n                    if time.monotonic() < stop_submitting_at:\n                        submit_one(executor)\n                    else:\n                        stopped_for_deadline = True\n\n                    if completed % checkpoint_every == 0:\n                        state = {\n                            **base_state,\n                            "updated_at_utc": utc_now(),\n                            "new_tasks_submitted": submitted,\n                            "new_tasks_completed": completed,\n                            "successful_or_resumed_videos": len(metadata_rows),\n                            "failed_videos": len(failure_rows),\n                            "stopped_for_deadline": stopped_for_deadline,\n                        }\n                        checkpoint(\n                            metadata_rows,\n                            failure_rows,\n                            metadata_path,\n                            failure_path,\n                            state_path,\n                            state,\n                        )\n\n    return completed, stopped_for_deadline\n\n\ndef main(config_path: Path) -> None:\n    run_started = time.monotonic()\n\n    # Fail immediately and legibly if MediaPipe cannot import. Without this\n    # the failure surfaces only inside workers, as \'every benchmark failed\'.\n    try:\n        from mediapipe.python.solutions import holistic as _import_check\n        del _import_check\n    except Exception as error:\n        raise RuntimeError(\n            f\'MediaPipe cannot be imported: {type(error).__name__}: {error}\\n\'\n            \'Run the install/patch cell and restart the kernel.\'\n        ) from error\n    with config_path.open("r", encoding="utf-8") as handle:\n        config = json.load(handle)\n\n    shard_count = int(config["shard_count"])\n    shard_index = int(config["shard_index"])\n    if shard_count < 1:\n        raise ValueError("shard_count must be at least 1")\n    if not 0 <= shard_index < shard_count:\n        raise ValueError(\n            f"shard_index must be from 0 to {shard_count - 1}; got {shard_index}"\n        )\n\n    shard_tag = f"shard_{shard_index:03d}_of_{shard_count:03d}"\n    output_base = Path(config["output_base"])\n    shard_output_root = output_base / shard_tag\n    shard_output_root.mkdir(parents=True, exist_ok=True)\n\n    metadata_path = Path("/kaggle/working") / f"holistic_metadata_{shard_tag}.csv"\n    failure_path = Path("/kaggle/working") / f"holistic_failures_{shard_tag}.csv"\n    state_path = Path("/kaggle/working") / f"holistic_state_{shard_tag}.json"\n    manifest_path = Path("/kaggle/working") / f"holistic_manifest_{shard_tag}.json"\n    benchmark_path = Path("/kaggle/working") / f"holistic_benchmark_{shard_tag}.json"\n    schema_path = Path("/kaggle/working/holistic_feature_schema.json")\n\n    max_run_seconds = float(config["max_run_hours"]) * 3600\n    reserve_seconds = float(config["finalization_reserve_minutes"]) * 60\n    stop_submitting_at = run_started + max_run_seconds - reserve_seconds\n\n    print("=" * 72)\n    print("Kaggle-safe MediaPipe Holistic extraction")\n    print("=" * 72)\n    print(f"Shard: {shard_index + 1}/{shard_count} ({shard_tag})")\n    print(f"Maximum script runtime: {config[\'max_run_hours\']} hours")\n    print(\n        "No new video will be submitted during the final "\n        f"{config[\'finalization_reserve_minutes\']} minutes."\n    )\n\n    video_root = discover_video_root(config)\n    all_videos = discover_videos(video_root)\n    shard_videos = all_videos[shard_index::shard_count]\n\n    print(f"Video root: {video_root}")\n    print(f"All videos discovered: {len(all_videos):,}")\n    print(f"Videos assigned to this shard: {len(shard_videos):,}")\n    print("Class inference check - confirm these are word/class folders:")\n    for sample in evenly_spaced_items(shard_videos, 5):\n        sample_relative = sample.relative_to(video_root)\n        print(\n            f"  {sample_relative.as_posix()}  ->  "\n            f"class={class_from_relative_path(sample_relative)}"\n        )\n    print(f"Output directory: {shard_output_root}")\n\n    if not shard_videos:\n        raise RuntimeError("This shard contains no videos. Reduce shard_count.")\n\n    resume_source = locate_resume_source(\n        config,\n        shard_tag,\n        shard_output_root,\n    )\n    if resume_source is not None:\n        copy_resume_output(resume_source, shard_output_root)\n\n    # Remove abandoned temporary files. Final .npy files are never touched here.\n    for partial in shard_output_root.rglob("*.partial.*"):\n        partial.unlink(missing_ok=True)\n\n    detector_config = {\n        "model_complexity": int(config["model_complexity"]),\n        "min_detection_confidence": float(config["min_detection_confidence"]),\n        "min_tracking_confidence": float(config["min_tracking_confidence"]),\n    }\n\n    feature_schema = {\n        "total_landmarks": TOTAL_LANDMARKS,\n        "coordinates_per_landmark": COORDINATES_PER_LANDMARK,\n        "features_per_frame": FEATURES_PER_FRAME,\n        "dtype": "float32",\n        "missing_value_strategy": "zero padding by modality",\n        "fresh_holistic_graph_per_video": True,\n        "feature_order": [\n            {"modality": "face", "landmarks": FACE_LANDMARKS, "start": 0, "end": FACE_SLICE.stop},\n            {"modality": "pose", "landmarks": POSE_LANDMARKS, "start": POSE_SLICE.start, "end": POSE_SLICE.stop},\n            {"modality": "left_hand", "landmarks": LEFT_HAND_LANDMARKS, "start": LEFT_HAND_SLICE.start, "end": LEFT_HAND_SLICE.stop},\n            {"modality": "right_hand", "landmarks": RIGHT_HAND_LANDMARKS, "start": RIGHT_HAND_SLICE.start, "end": RIGHT_HAND_SLICE.stop},\n        ],\n        "detector_config": detector_config,\n    }\n    atomic_write_json(schema_path, feature_schema)\n\n    metadata_rows: List[Dict[str, Any]] = []\n    failure_rows: List[Dict[str, Any]] = []\n    remaining_tasks: List[Dict[str, Any]] = []\n\n    overwrite = bool(config.get("overwrite_existing", False))\n    for video_path in tqdm(shard_videos, desc="Checking existing output", unit="video"):\n        task = make_task(\n            video_path,\n            video_root,\n            shard_output_root,\n            detector_config,\n            save_output=True,\n        )\n        output_path = Path(task["output_path"])\n        existing = None if overwrite else valid_existing_sequence(output_path)\n\n        if existing is not None:\n            metadata_rows.append(\n                {\n                    "video_path": task["video_path"],\n                    "relative_video_path": task["relative_video_path"],\n                    "class": task["class"],\n                    "landmark_file": task["output_path"],\n                    "landmark_file_relative": task["landmark_file_relative"],\n                    **existing,\n                    "status": "resumed_valid_existing",\n                }\n            )\n        else:\n            remaining_tasks.append(task)\n\n    prior_statistics: Dict[str, Dict[str, Any]] = {}\n    for prior_csv in sorted(\n        Path(str(config.get("input_root", "/kaggle/input"))).rglob(\n            f"holistic_metadata_{shard_tag}.csv"\n        )\n    ):\n        try:\n            prior_frame = pd.read_csv(prior_csv)\n        except Exception:\n            continue\n        for record in prior_frame.to_dict("records"):\n            key = str(record.get("relative_video_path", ""))\n            if key:\n                prior_statistics[key] = record\n\n    if prior_statistics:\n        carried = 0\n        for row in metadata_rows:\n            record = prior_statistics.get(row.get("relative_video_path", ""))\n            if not record:\n                continue\n            for column in (\n                "face_detection_rate",\n                "pose_detection_rate",\n                "left_hand_detection_rate",\n                "right_hand_detection_rate",\n                "mean_pose_visibility",\n                "fps",\n                "width",\n                "height",\n                "declared_frame_count",\n            ):\n                value = record.get(column)\n                if value is not None and pd.notna(value):\n                    row[column] = value\n            carried += 1\n        print(f"Carried QC statistics forward for {carried:,} resumed videos.")\n\n    print(f"Valid existing outputs: {len(metadata_rows):,}")\n    print(f"Videos still requiring extraction: {len(remaining_tasks):,}")\n\n    if not remaining_tasks:\n        chosen_workers = 0\n        benchmark_reports: List[Dict[str, Any]] = []\n        best_benchmark: Dict[str, Any] = {}\n    else:\n        benchmark_paths = evenly_spaced_items(\n            [Path(task["video_path"]) for task in remaining_tasks],\n            int(config["benchmark_videos"]),\n        )\n        benchmark_tasks = [\n            make_task(\n                path,\n                video_root,\n                shard_output_root,\n                detector_config,\n                save_output=False,\n            )\n            for path in benchmark_paths\n        ]\n\n        cpu_count = max(1, os.cpu_count() or 1)\n        configured_workers = int(config.get("max_workers", 0))\n        if configured_workers > 0:\n            worker_candidates = [min(configured_workers, cpu_count)]\n        else:\n            auto_cap = min(int(config.get("auto_worker_cap", 4)), cpu_count)\n            worker_candidates = list(range(1, auto_cap + 1))\n\n        chosen_workers, benchmark_reports, best_benchmark = benchmark_workers(\n            benchmark_tasks,\n            worker_candidates,\n        )\n\n        seconds_per_video = float(best_benchmark["seconds_per_video_effective"])\n        raw_projected_seconds = seconds_per_video * len(remaining_tasks)\n        safe_projected_seconds = raw_projected_seconds * float(\n            config["runtime_safety_factor"]\n        )\n        processing_seconds_available = max(0.0, stop_submitting_at - time.monotonic())\n\n        mean_output_bytes = float(best_benchmark["mean_output_bytes_per_video"])\n        projected_remaining_bytes = (\n            mean_output_bytes\n            * len(remaining_tasks)\n            * float(config["output_size_safety_factor"])\n        )\n        free_bytes = shutil.disk_usage("/kaggle/working").free\n\n        preflight = {\n            "created_at_utc": utc_now(),\n            "shard": shard_tag,\n            "video_root": str(video_root),\n            "all_videos": len(all_videos),\n            "videos_in_shard": len(shard_videos),\n            "valid_existing_outputs": len(metadata_rows),\n            "remaining_videos": len(remaining_tasks),\n            "worker_candidates": worker_candidates,\n            "chosen_workers": chosen_workers,\n            "benchmark_reports": benchmark_reports,\n            "raw_projected_hours": raw_projected_seconds / 3600,\n            "safe_projected_hours": safe_projected_seconds / 3600,\n            "processing_hours_available": processing_seconds_available / 3600,\n            "projected_remaining_output_gib": projected_remaining_bytes / (1024 ** 3),\n            "free_working_space_gib": free_bytes / (1024 ** 3),\n        }\n        atomic_write_json(benchmark_path, preflight)\n        print("\\nPreflight result")\n        print(json.dumps(preflight, indent=2))\n\n        if projected_remaining_bytes > free_bytes * 0.90:\n            raise RuntimeError(\n                "Projected output may exceed safe /kaggle/working capacity. "\n                "Increase shard_count and run separate shard versions."\n            )\n\n        if (\n            bool(config.get("abort_if_projected_too_long", True))\n            and safe_projected_seconds > processing_seconds_available\n        ):\n            recommended_total_shards = max(\n                shard_count + 1,\n                math.ceil(\n                    safe_projected_seconds\n                    * shard_count\n                    / max(processing_seconds_available * 0.90, 1.0)\n                ),\n            )\n            raise RuntimeError(\n                "The conservative benchmark does not fit the safe runtime window.\\n"\n                f"Recommended shard_count: at least {recommended_total_shards}.\\n"\n                "Set shard_count to that value and run one saved Kaggle version "\n                "for every shard_index from 0 to shard_count - 1."\n            )\n\n    base_state = {\n        "started_at_utc": utc_now(),\n        "shard": shard_tag,\n        "video_root": str(video_root),\n        "all_videos_discovered": len(all_videos),\n        "videos_in_shard": len(shard_videos),\n        "valid_existing_at_start": len(metadata_rows),\n        "new_videos_required": len(remaining_tasks),\n        "chosen_workers": chosen_workers,\n    }\n\n    completed_new, stopped_for_deadline = run_extraction(\n        tasks=remaining_tasks,\n        workers=max(1, chosen_workers),\n        stop_submitting_at=stop_submitting_at,\n        checkpoint_every=int(config["checkpoint_every"]),\n        metadata_rows=metadata_rows,\n        failure_rows=failure_rows,\n        metadata_path=metadata_path,\n        failure_path=failure_path,\n        state_path=state_path,\n        base_state=base_state,\n    )\n\n    metadata_rows.sort(key=lambda row: row.get("relative_video_path", ""))\n    failure_rows.sort(key=lambda row: row.get("relative_video_path", ""))\n\n    accounted_paths = {\n        row.get("relative_video_path", "")\n        for row in metadata_rows + failure_rows\n        if row.get("relative_video_path", "")\n    }\n    videos_not_attempted = len(shard_videos) - len(accounted_paths)\n    run_complete = videos_not_attempted == 0 and not stopped_for_deadline\n\n    # Verify a representative subset of final arrays.\n    verification_rows = evenly_spaced_items(metadata_rows, min(25, len(metadata_rows)))\n    verification_failures: List[Dict[str, Any]] = []\n    for row in verification_rows:\n        path = Path(row["landmark_file"])\n        valid = valid_existing_sequence(path)\n        if valid is None:\n            verification_failures.append(\n                {\n                    "relative_video_path": row.get("relative_video_path", ""),\n                    "landmark_file": str(path),\n                }\n            )\n\n    state = {\n        **base_state,\n        "updated_at_utc": utc_now(),\n        "new_tasks_completed": completed_new,\n        "successful_or_resumed_videos": len(metadata_rows),\n        "failed_videos": len(failure_rows),\n        "videos_not_attempted": videos_not_attempted,\n        "stopped_for_deadline": stopped_for_deadline,\n        "run_complete": run_complete,\n    }\n    checkpoint(\n        metadata_rows,\n        failure_rows,\n        metadata_path,\n        failure_path,\n        state_path,\n        state,\n    )\n\n    metadata_frame = pd.DataFrame(metadata_rows)\n    total_frames = (\n        int(metadata_frame["frames"].fillna(0).sum())\n        if not metadata_frame.empty and "frames" in metadata_frame\n        else 0\n    )\n\n    manifest = {\n        **state,\n        "finished_at_utc": utc_now(),\n        "elapsed_hours": (time.monotonic() - run_started) / 3600,\n        "features_per_frame": FEATURES_PER_FRAME,\n        "total_frames": total_frames,\n        "videos_recovered_by_retry": sum(\n            1 for row in metadata_rows if row.get("recovered_after_error")\n        ),\n        "verification_files_checked": len(verification_rows),\n        "verification_failures": verification_failures,\n        "metadata_csv": str(metadata_path),\n        "failure_csv": str(failure_path),\n        "state_json": str(state_path),\n        "benchmark_json": str(benchmark_path),\n        "feature_schema_json": str(schema_path),\n        "output_directory": str(shard_output_root),\n    }\n    atomic_write_json(manifest_path, manifest)\n\n    print("\\n" + "=" * 72)\n    print("FINAL RUN MANIFEST")\n    print("=" * 72)\n    print(json.dumps(manifest, indent=2))\n\n    if verification_failures:\n        raise RuntimeError(\n            "Final verification found invalid saved arrays. Review the manifest."\n        )\n\n    if run_complete:\n        print("\\nSUCCESS: every video assigned to this shard was accounted for.")\n    else:\n        print(\n            "\\nSAFE PARTIAL FINISH: Kaggle should save the completed arrays. "\n            "Attach this version\'s output as an input and rerun the same shard "\n            "to resume."\n        )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", type=Path, required=True)\n    arguments = parser.parse_args()\n    main(arguments.config)\n'
RUNNER_PATH.write_text(RUNNER_CODE, encoding="utf-8")
print(f"Runner written to: {RUNNER_PATH} ({len(RUNNER_CODE):,} bytes)")

In [ ]:
# ============================================================
# CELL 4 - Smoke test (about one minute). Run this before committing.
# ============================================================
# Checks the three things that have gone wrong before:
#   1. MediaPipe loads inside a background worker
#   2. The videos are found and labelled from the right folder
#   3. Workers do not fail the first video they touch

import concurrent.futures as cf
import json
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
import run_holistic_extraction as runner

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
video_root = runner.discover_video_root(config)
videos = runner.discover_videos(video_root)

print(f"Video root: {video_root}")
print(f"Videos found: {len(videos):,}\n")

# LABEL CHECK - the single most important line in this cell.
# Every video must get its own word as a label. If every label comes out
# the same, extraction would still "work" but the labels would be useless.
all_labels = [
    runner.class_from_relative_path(path.relative_to(video_root))
    for path in videos
]
unique_labels = sorted(set(all_labels))

print(f"Distinct labels found: {len(unique_labels)}")
print(f"Videos per label: about {len(videos) // max(len(unique_labels), 1)}")
print(f"First few labels: {unique_labels[:8]}")
print()
for sample in runner.evenly_spaced_items(videos, 5):
    relative = sample.relative_to(video_root)
    print(f"  {relative.as_posix()}  ->  label = {runner.class_from_relative_path(relative)}")

if len(unique_labels) < 2:
    raise RuntimeError(
        f"Only {len(unique_labels)} label found ({unique_labels}). Every video "
        "would belong to one class and training could not work. Stop here."
    )
print(f"\nLabels look right: {len(unique_labels)} different words.")

detector_config = {
    "model_complexity": int(config["model_complexity"]),
    "min_detection_confidence": float(config["min_detection_confidence"]),
    "min_tracking_confidence": float(config["min_tracking_confidence"]),
}

tasks = [
    runner.make_task(
        path, video_root, Path("/kaggle/working/_smoketest"),
        detector_config, save_output=False,
    )
    for path in runner.evenly_spaced_items(videos, 6)
]

print(f"\nProcessing {len(tasks)} videos across 2 background workers...")
results = []
with cf.ProcessPoolExecutor(**runner.executor_kwargs(2)) as executor:
    for result in executor.map(runner.process_video_task, tasks):
        results.append(result)

successes = [r for r in results if r.get("ok")]
failures = [r for r in results if not r.get("ok")]
recovered = [r for r in successes if r.get("recovered_after_error")]

print(f"\nSucceeded: {len(successes)}   Failed: {len(failures)}   "
      f"Needed a retry: {len(recovered)}")

for result in recovered:
    print(f"  retried and worked: {Path(result['video_path']).name}")
    print(f"    first attempt said: {result['recovered_after_error']}")

for result in failures:
    print("\n" + "=" * 72)
    print(f"VIDEO: {result['video_path']}")
    print(f"ERROR: {result['error_type']}: {result['error_message']}")
    print(result.get("traceback", ""))

if successes:
    frames = [r["frames"] for r in successes]
    print(f"\nFrames per video: {min(frames)} to {max(frames)}")
    print(f"Numbers per frame: {successes[0]['features']} (must be 1629)")
    assert successes[0]["features"] == 1629

print()
if failures:
    print("STOP. Read the error above before committing a run this long.")
else:
    print("Smoke test clean. Go ahead: Save Version -> Save & Run All.")

In [ ]:
# ============================================================
# CELL 5 - The actual extraction. This is the long one.
# ============================================================
# Once you commit, this keeps running on Kaggle's computer even after you
# close the browser, lose wifi, or lose power.
#
# We deliberately do NOT crash the notebook if this exits with an error.
# A version that ends in a crash gets marked failed, and a failed version
# is not a dependable way to hand you back 10 hours of finished files.

import subprocess
import sys

command = [sys.executable, str(RUNNER_PATH), "--config", str(CONFIG_PATH)]
print("Running:", " ".join(command), flush=True)

completed = subprocess.run(command)
EXTRACTION_RETURN_CODE = completed.returncode

if EXTRACTION_RETURN_CODE == 0:
    print("\nExtraction finished cleanly.")
else:
    print(f"\nExtraction exited with code {EXTRACTION_RETURN_CODE}.")
    print("Everything it completed is still saved. See the report below.")

In [ ]:
# ============================================================
# CELL 6 - Report: did it finish, and what do I have?
# ============================================================
import json
from pathlib import Path

shard_tag = f"shard_{CONFIG['shard_index']:03d}_of_{CONFIG['shard_count']:03d}"
manifest_path = Path(f"/kaggle/working/holistic_manifest_{shard_tag}.json")

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(manifest, indent=2))

    print("\n" + "=" * 72)
    if manifest.get("run_complete"):
        print("COMPLETE. All videos are done. Move on to training.")
    else:
        done = manifest.get("successful_or_resumed_videos", 0)
        total = manifest.get("videos_in_shard", 0)
        print(f"NOT FINISHED: {done:,} of {total:,} videos done.")
        print("To finish the rest:")
        print("  1. Add Data -> Your Work -> Notebooks -> this version's output")
        print("  2. Change nothing else")
        print("  3. Save Version -> Save & Run All")
else:
    print(f"No manifest at {manifest_path}.")
    print("It stopped before writing the summary. Check the log above.")
    state_path = Path(f"/kaggle/working/holistic_state_{shard_tag}.json")
    if state_path.exists():
        print(json.dumps(json.loads(state_path.read_text(encoding="utf-8")), indent=2))

output_root = Path(CONFIG["output_base"])
npy_count = sum(1 for _ in output_root.rglob("*.npy")) if output_root.exists() else 0
total_bytes = sum(p.stat().st_size for p in Path("/kaggle/working").rglob("*") if p.is_file())

print(f"\nNumber files saved: {npy_count:,} (target 6,000)")
print(f"Space used: {total_bytes / (1024 ** 3):.2f} GB of the 20 GB limit")

for path in sorted(Path("/kaggle/working").glob("holistic_*")):
    if path.is_file():
        print(f"  {path.name}")